In [ ]:
pip install torch


  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-m

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split


In [ ]:
df= pd.read_csv('/content/drive/MyDrive/ireland task/3 dataset task/Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements.csv')

df.head()

,COMMODITY,COMMODITY_DESCRIPTION,EXTENDED_DESCRIPTION,QUANTITY,UNIT_OF_MEASURE,UNIT_OF_MEAS_DESC,UNIT_PRICE,ITM_TOT_AM,MASTER_AGREEMENT,CONTRACT_NAME,...,AWARD_DATE,VENDOR_CODE,LGL_NM,AD_LN_1,AD_LN_2,CITY,ST,ZIP,CTRY,DATA_BUILD_DATE
0,8853830,"CHLORINE, LIQUID (FOR WATER PURIFICATION)",NaN,10.0,CYL,Cylinder,577.80,5778.00,MA2200GA190000042,DAVIS WTP- DXI-LIQUID CHLORINE-FEBRUARY 2020,...,01/31/2020,DXI7126645,DXI INDUSTRIES INC,1919 JACINTOPORT BLVD,NaN,HOUSTON,TX,77015-6585,US,03/18/2024
1,25726,CBRNE Medical Supplies and Pharmaceuticals (No...,PR1625-Cool Cube vaccine carrier for Wellness ...,0.0,NaN,NaN,0.00,2477.00,NaN,PR1625-Cool Cube vaccine carrier for Wellness ...,...,08/25/2016,V00000939384,"VeriCor, LLC",703 Western Ave,NaN,Holmen,WI,54636,US,03/18/2024
2,2008413,JACKET,Old PO# 16092204609,2.0,EA,Each,54.99,109.98,NaN,"Jackets for Administration, Finance and IT MAX...",...,10/25/2016,ARA0282750,ARAMARK UNIFORM SERVICES INC,1057 SOLUTIONS CENTER,NaN,CHICAGO,IL,60677-1000,US,03/18/2024
3,84070,"Video Camera-Recorders, Accessories and Parts ...",SHAPE FOLLOW FOCUS SHFFCLIC / FFCLIC,1.0,EA,Each,373.95,373.95,NaN,CHANNELAUSTINSTUDIO & FIELD VIDEO PACKAGE,...,07/17/2014,BHP6052500,B & H FOTO & ELECTRONICS CORP,420 9TH AVE,NaN,NEW YORK,NY,10001-1603,US,03/18/2024
4,2803055,"CABLE, POWER CONTROL",JB7174 - End connector,5.0,EA,Each,240.00,1200.00,MA8100GS090000011,POWER CONTROL CABLE ASSY,...,01/03/2011,VC0000102891,GSE HOLDINGS INC,907 COTTING LANE STE A,NaN,VACAOILLE,CA,95688,US,03/18/2024


In [ ]:
import pandas as pd
import torch

# Load the dataset
df = pd.read_csv("/content/drive/MyDrive/ireland task/3 dataset task/Purchase_Order_Quantity_Price_detail_for_Commodity_Goods_procurements.csv")

# Fill missing values in numeric columns with 0
numeric_columns = df.select_dtypes(include=['float64', 'float32', 'float16', 'int64', 'int32', 'int16', 'int8']).columns
df[numeric_columns] = df[numeric_columns].fillna(0)

# Label encode categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns
label_encoders = {}
for col in categorical_columns:
    labels, uniques = pd.factorize(df[col])
    label_encoders[col] = dict(zip(uniques, range(len(uniques))))
    df[col] = torch.tensor(labels)

# Convert bool columns to integers
bool_columns = df.select_dtypes(include=['bool']).columns
df[bool_columns] = df[bool_columns].astype(int)

# Now, all columns should be numeric
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291065 entries, 0 to 291064
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   COMMODITY              291065 non-null  int64  
 1   COMMODITY_DESCRIPTION  291065 non-null  int64  
 2   EXTENDED_DESCRIPTION   291065 non-null  int64  
 3   QUANTITY               291065 non-null  float64
 4   UNIT_OF_MEASURE        291065 non-null  int64  
 5   UNIT_OF_MEAS_DESC      291065 non-null  int64  
 6   UNIT_PRICE             291065 non-null  float64
 7   ITM_TOT_AM             291065 non-null  float64
 8   MASTER_AGREEMENT       291065 non-null  int64  
 9   CONTRACT_NAME          291065 non-null  int64  
 10  PURCHASE_ORDER         291065 non-null  int64  
 11  AWARD_DATE             291065 non-null  int64  
 12  VENDOR_CODE            291065 non-null  int64  
 13  LGL_NM                 291065 non-null  int64  
 14  AD_LN_1                291065 non-nu

In [ ]:
import pandas as pd
import torch



# Convert DataFrame to PyTorch tensors
data_tensor = torch.tensor(df.values)

# Check for null values
null_tensor = torch.isnan(data_tensor[:, :]).any()
print("Null values present:", null_tensor.item())




Null values present: False


In [ ]:
!pip install skorch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.3/239.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
# Convert dataframe to PyTorch tensors
X = torch.tensor(df.drop('PURCHASE_ORDER', axis=1).values, dtype=torch.float32)
y = torch.tensor(df['PURCHASE_ORDER'].values, dtype=torch.float32)

# Create a dataset and dataloaders
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
class DecisionTree:
    def __init__(self):
        pass

    def fit(self, X, y):
        # Implement the fitting logic
        pass

    def predict(self, X):
        # Implement the prediction logic
        pass

class RandomForest:
    def __init__(self, n_trees=10, max_features=None):
        self.trees = [DecisionTree() for _ in range(n_trees)]
        self.max_features = max_features

    def fit(self, X, y):
        for tree in self.trees:
            # Bootstrap sample
            indices = torch.randperm(len(y))
            bootstrap_X = X[indices]
            bootstrap_y = y[indices]
            # Feature sampling
            if self.max_features is not None:
                feat_indices = torch.randperm(X.shape[1])[:self.max_features]
                bootstrap_X = bootstrap_X[:, feat_indices]
            tree.fit(bootstrap_X, bootstrap_y)

    def predict(self, X):
        # Collect predictions from each tree
        predictions = [tree.predict(X) for tree in self.trees]
        # Majority voting
        predictions = torch.stack(predictions, dim=0)
        result, _ = torch.mode(predictions, dim=0)
        return result

# Example usage
rf = RandomForest(n_trees=100, max_features=int(sqrt(X_train.shape[1])))
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)


In [ ]:
import torch

class DecisionTreeNode:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_index = feature_index  # Index of the feature used for splitting
        self.threshold = threshold  # Threshold value for splitting
        self.left = left  # Left child
        self.right = right  # Right child
        self.value = value  # Value if the node is a leaf node

    def is_leaf_node(self):
        return self.value is not None


In [ ]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=5):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None

    def fit(self, features, target):
        self.root = self._build_tree(features, target)

    def _build_tree(self, features, target, depth=0):
        num_samples, num_features = features.shape
        # Stopping conditions
        if depth >= self.max_depth or num_samples < self.min_samples_split:
            leaf_value = self._calculate_leaf_value(target)
            return DecisionTreeNode(value=leaf_value)

        # Find the best split
        best_feature, best_threshold = self._find_best_split(features, target, num_features)

        # Grow the left and right subtrees recursively
        left_indices, right_indices = self._split(features[:, best_feature], best_threshold)
        left_subtree = self._build_tree(features[left_indices], target[left_indices], depth + 1)
        right_subtree = self._build_tree(features[right_indices], target[right_indices], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left_subtree, right_subtree)

    def _calculate_leaf_value(self, target):
        # For regression, return mean; for classification, return mode
        return target.mean()

    def _find_best_split(self, features, target, num_features):
        # Implement logic to find the best feature and threshold to split on
        pass

    def _split(self, feature_values, threshold):
        # Implement logic to split data
        pass

    def predict(self, features):
        # Navigate the tree for each feature set and return predictions
        pass


In [ ]:
class RandomForest:
    def __init__(self, n_trees=10, min_samples_split=2, max_depth=5):
        self.trees = [DecisionTree(min_samples_split, max_depth) for _ in range(n_trees)]

    def fit(self, features, target):
        for tree in self.trees:
            # Implement bootstrap sampling
            indices = torch.randperm(len(target))
            bootstrap_features = features[indices]
            bootstrap_target = target[indices]
            tree.fit(bootstrap_features, bootstrap_target)

    def predict(self, features):
        # Aggregate predictions from all trees
        tree_preds = [tree.predict(features) for tree in self.trees]
        tree_preds = torch.stack(tree_preds)
        # For classification, take the mode; for regression, take the mean
        return tree_preds.mode(0).values  # or tree_preds.mean(0) for regression


In [ ]:
import torch
from sklearn.metrics import classification_report

# Setup and train model
random_forest = RandomForest(n_trees=10, min_samples_split=2, max_depth=5)
random_forest.fit(X_train, y_train)

# Predictions
predictions = random_forest.predict(X_test)

# Calculate accuracy
correct = (predictions == y_test).float().sum()
accuracy = correct / len(y_test)
print(f"Accuracy: {accuracy:.4f}")

# Classification report
y_test_np = y_test.cpu().numpy()  # Assuming y_test is a tensor
predictions_np = predictions.cpu().numpy()  # Convert predictions to numpy for sklearn
report = classification_report(y_test_np, predictions_np)
print("Classification Report:")
print(report)


In [ ]:
print(accuracy.score(train, train_labels))
print(accuracy.score(test, test_labels))

0.9878542510121457
0.8870967741935484


In [ ]:


# Classification report
print(classification_report(test_labels, predictions))


              precision    recall  f1-score   support

           0       0.60      0.67      0.63         9
           1       0.94      0.92      0.93        53

    accuracy                           0.89        62
   macro avg       0.77      0.80      0.78        62
weighted avg       0.89      0.89      0.89        62



In [ ]:
import torch

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = torch.tensor(X, dtype=torch.float32)
        self.y_train = torch.tensor(y, dtype=torch.long)

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        distances = torch.cdist(X, self.X_train)
        _, indices = torch.topk(distances, self.k, largest=False)
        votes = torch.mode(self.y_train[indices], dim=1).values
        return votes.numpy()

# Create KNN instance and predict
knn = KNN(k=3)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)


<ipython-input-44-2a6f17b3090f>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X_train = torch.tensor(X, dtype=torch.float32)
<ipython-input-44-2a6f17b3090f>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y_train = torch.tensor(y, dtype=torch.long)
<ipython-input-44-2a6f17b3090f>:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)


In [ ]:
def knn_predict_batch(X_train, y_train, X_test, k, batch_size=64):
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_test = torch.tensor(X_test, dtype=torch.float32)

    y_pred = []
    # Split X_test into batches
    total = X_test.shape[0]
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = X_test[start:end]
        # Compute distances to all training points
        distances = torch.cdist(batch, X_train, p=2)  # p=2 for Euclidean distance
        # Find the k nearest neighbors and their labels
        _, indices = torch.topk(distances, k, dim=1, largest=False)
        neighbor_labels = y_train[indices]
        # Majority vote
        preds, _ = torch.mode(neighbor_labels, dim=1)
        y_pred.extend(preds.tolist())

    return torch.tensor(y_pred)

# Predict using KNN with batching
y_pred_knn_batch = knn_predict_batch(X_train, y_train, X_test, k=5)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Convert tensors to NumPy arrays if necessary
y_test = torch.tensor(y_test, dtype=torch.long).numpy()  # Ensure y_test is a numpy array
y_pred_knn_batch = y_pred_knn_batch.numpy()  # Convert predictions to numpy array if they are in tensor format

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred_knn_batch)
print(f"Accuracy: {accuracy:.4f}")

# Generate a classification report
report = classification_report(y_test, y_pred_knn_batch)
print("Classification Report:")
print(report)


In [ ]:
print(report.score(train, train_labels))
print(report.score(test, test_labels))

0.8987854251012146
0.8709677419354839


In [ ]:


# Classification report
print(classification_report(test_labels, predictions))


              precision    recall  f1-score   support

           0       0.67      0.22      0.33         9
           1       0.88      0.98      0.93        53

    accuracy                           0.87        62
   macro avg       0.77      0.60      0.63        62
weighted avg       0.85      0.87      0.84        62

